# Part F: Permutation Invariant Training (PIT)

---

## The Speaker Ordering Problem

Your `MiniEEND` outputs `(batch, T, 3)` — three probability streams, one per speaker slot. Let's say for a clip with Speaker A and Speaker B, the output looks like:

```
Output slot 0:  [0.9, 0.8, 0.1, 0.1, ...]  ← active early
Output slot 1:  [0.1, 0.1, 0.8, 0.9, ...]  ← active late
Output slot 2:  [0.1, 0.1, 0.1, 0.1, ...]  ← mostly silent
```

This is a good prediction — one speaker talks early, another talks late. But now you need to compute a loss. Your ground truth labels look like:

```
Speaker A:  [0, 0, 1, 1, ...]  ← active late
Speaker B:  [1, 1, 0, 0, ...]  ← active early
```

Which output slot should be compared to Speaker A? Which to Speaker B?

The network assigned slot 0 to the early speaker and slot 1 to the late speaker. Your ground truth has Speaker A as late and Speaker B as early. If you naively compare slot 0 → Speaker A and slot 1 → Speaker B, the loss will be enormous — even though the prediction is actually correct.

This is the **speaker ordering problem**: the network has no reason to output speakers in any particular order, and the ground truth labels have no canonical order either. There's no rule that says "Speaker A always goes in slot 0."

---

## Why This Is Fundamental, Not a Detail

You might think: just always sort speakers by when they first appear. Speaker B speaks first, so put B in slot 0.

The problem: during training, the network is learning. In the middle of training, its outputs are messy — it might not have a clear "first speaker" at all. Imposing an ordering rule on noisy outputs creates inconsistent training signal.

More fundamentally: in a real recording you don't know who is Speaker A and who is Speaker B. Those labels are arbitrary. The ground truth file might have been generated with speakers assigned randomly. The network should not be penalized for learning the right *pattern* in the wrong *slot*.

---

## How PIT Works

**Permutation Invariant Training** solves this by trying all possible assignments between output slots and ground truth speakers, and using whichever assignment gives the lowest loss.

For 3 speakers, there are `3! = 6` possible assignments:

```
Permutation 0: slot 0 → spk A,  slot 1 → spk B,  slot 2 → spk C
Permutation 1: slot 0 → spk A,  slot 1 → spk C,  slot 2 → spk B
Permutation 2: slot 0 → spk B,  slot 1 → spk A,  slot 2 → spk C
Permutation 3: slot 0 → spk B,  slot 1 → spk C,  slot 2 → spk A
Permutation 4: slot 0 → spk C,  slot 1 → spk A,  slot 2 → spk B
Permutation 5: slot 0 → spk C,  slot 1 → spk B,  slot 2 → spk A
```

For each permutation, compute the loss between the permuted predictions and ground truth. Take the minimum:

```
PIT loss = min over all permutations( loss(predictions[permutation], ground_truth) )
```

This minimum loss and its gradients flow back through the network. The network learns: "produce this pattern somewhere in your output slots — I don't care which slot."

---

## A Concrete Example

Predictions `(T=4, 2 speakers for simplicity)`:
```
slot 0: [0.9, 0.8, 0.1, 0.1]
slot 1: [0.1, 0.2, 0.8, 0.9]
```

Ground truth:
```
spk A:  [0, 0, 1, 1]
spk B:  [1, 1, 0, 0]
```

**Permutation 0:** slot 0 → spk A, slot 1 → spk B
```
BCE(slot 0, spk A) = BCE([0.9,0.8,0.1,0.1], [0,0,1,1]) → large loss
BCE(slot 1, spk B) = BCE([0.1,0.2,0.8,0.9], [1,1,0,0]) → large loss
Total: high
```

**Permutation 1:** slot 0 → spk B, slot 1 → spk A
```
BCE(slot 0, spk B) = BCE([0.9,0.8,0.1,0.1], [1,1,0,0]) → small loss ✓
BCE(slot 1, spk A) = BCE([0.1,0.2,0.8,0.9], [0,0,1,1]) → small loss ✓
Total: low
```

PIT picks Permutation 1. The gradient tells the network: "slot 0 should be tracking Speaker B, slot 1 should be tracking Speaker A." The network adjusts accordingly.

---

## PIT Loss Calculation in Practice

The algorithm:

```
1. Compute loss for every (output slot, ground truth speaker) pair
   → This gives you a cost matrix of shape (num_speakers, num_speakers)

2. Find the permutation (assignment) that minimizes total cost
   → For small N (2-4 speakers): enumerate all N! permutations
   → For large N: use the Hungarian algorithm (optimal assignment in O(N³))

3. Use the minimum-cost assignment's loss for backpropagation
```

The **cost matrix** for 3 speakers looks like:

```
              spk A    spk B    spk C
slot 0:      [ 0.8,    0.1,     0.6  ]
slot 1:      [ 0.7,    0.9,     0.3  ]
slot 2:      [ 0.2,    0.7,     0.8  ]
```

Entry `[i, j]` = loss between output slot `i` and ground truth speaker `j`.

The optimal assignment picks one entry per row and per column (each slot assigned to exactly one speaker, each speaker assigned to exactly one slot) such that the total is minimized. Here: slot 0 → spk B (0.1), slot 1 → spk C (0.3), slot 2 → spk A (0.2). Total = 0.6.

For 2-3 speakers, you can just enumerate all permutations with `itertools.permutations`. For larger numbers, `scipy.optimize.linear_sum_assignment` implements the Hungarian algorithm.

---

## The Hungarian Algorithm (Conceptual)

You don't need to implement this from scratch — `scipy` provides it. But you should know what it does.

Given a cost matrix, Hungarian finds the assignment of rows to columns that minimizes total cost, in O(N³) time. For N=3, it's overkill (6 permutations is trivial). For N=10, enumeration would require 3,628,800 checks; Hungarian does it in under 1000 operations.

```python
from scipy.optimize import linear_sum_assignment
row_indices, col_indices = linear_sum_assignment(cost_matrix)
# row_indices: which output slots were selected (always [0,1,2,...])
# col_indices: which ground truth speaker each slot maps to
```

---

## When PIT Is Needed vs Not

This is the most important conceptual point of Part F:

**PIT is needed when:**
- Your network produces multiple output streams that could each correspond to any of N targets
- The assignment between outputs and targets is ambiguous
- Examples: speaker separation (which output is Speaker A?), multi-speaker diarization (which slot is which person?)

**PIT is NOT needed when:**
- There is only one output (binary: speech/silence)
- The output has a canonical order (class probabilities with fixed class meanings)
- You're doing counting/classification (your current `(batch, 3)` model — "how many speakers" has no ordering problem)

Your current speaker counter outputs `(batch, 3)` logits where class 0 = "1 speaker", class 1 = "2 speakers", class 2 = "3 speakers". These classes have fixed meanings — there's no ambiguity. Cross-entropy works fine.

Your `MiniEEND` outputs `(batch, T, 3)` where each of the 3 streams could correspond to any speaker. This needs PIT.

```
Task                          Needs PIT?
─────────────────────────────────────────
Count speakers (classify)     No  — fixed class meanings
Detect VAD (binary)           No  — one output
Separate N speakers           Yes — which output is which speaker?
Diarize N speakers            Yes — which slot is which person?
```

---

## TODO: Implement PIT Loss

**Goal:** Implement Permutation Invariant Training loss and use it to run one training step of your `MiniEEND`.

**Requirements:**

**Part 1 — Cost matrix:**

Write a function `compute_cost_matrix(predictions, targets)` where:
- `predictions`: `(num_speakers, T)` — the model's output for one sample, sigmoid already applied
- `targets`: `(num_speakers, T)` — ground truth binary activity for one sample
- Returns a `(num_speakers, num_speakers)` matrix where entry `[i, j]` = binary cross-entropy between `predictions[i]` and `targets[j]`

For BCE between two vectors use:
```
BCE(p, t) = -mean( t*log(p + 1e-8) + (1-t)*log(1-p + 1e-8) )
```

where p is prediction and t is ground truth or true label

**Part 2 — PIT loss for one sample:**

Write `pit_loss_single(predictions, targets)` that:
- Uses `compute_cost_matrix` to get the cost matrix
- Uses `scipy.optimize.linear_sum_assignment` to find the optimal assignment
- Returns the mean loss under the optimal assignment

**Part 3 — Batched PIT loss:**

Write `pit_loss_batch(predictions, targets)` that:
- Takes `predictions`: `(batch, T, num_speakers)` and `targets`: `(batch, T, num_speakers)`
- Loops over the batch, calls `pit_loss_single` for each sample
- Returns the mean loss across the batch

**Part 4 — One training step:**

Build a minimal training loop:
- Create synthetic ground truth: for each sample in a batch, randomly assign each of the 3 speakers a random binary activity pattern (use `torch.randint(0, 2, (T, 3)).float()`)
- Run `MiniEEND` forward pass
- Compute PIT loss
- Call `.backward()` and verify gradients exist on the model parameters

**Hints:**
- `linear_sum_assignment` minimizes cost — your cost matrix entries should be losses (positive numbers), not negative
- For Part 3, you'll need to permute dimensions: `predictions` comes out of the model as `(batch, T, num_speakers)` but `pit_loss_single` expects `(num_speakers, T)` — use `.permute()`
- The cost matrix should be built entirely with PyTorch operations so gradients flow through it
- `linear_sum_assignment` returns numpy arrays — you'll need to index your cost matrix with them to get the final loss: `cost_matrix[row_ind, col_ind].mean()`
- Verify gradients with: `any(p.grad is not None for p in model.parameters())`

**Key questions:**
- For 3 speakers, how many permutations does `linear_sum_assignment` evaluate internally vs brute force?
- Why must the cost matrix be computed with PyTorch operations rather than converting to numpy first?
- What would happen to training if you used the *maximum* cost permutation instead of minimum?

---

Start with Part 1. Show me `compute_cost_matrix` and a test run printing the `(3, 3)` cost matrix before moving to Part 2.

In [564]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import scipy
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as T
import math
from pathlib import Path
import numpy as np
import os
import sys
import json
import matplotlib.pyplot as plt

# Check if __file__ exists (it won't in Jupyter)
try: 
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [565]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        mixture_tensor = torch.from_numpy(mixture_audio)
        
        max_len = 480000 
        
        if mixture_tensor.size(0) > max_len:
            # Truncate if too long
            mixture_tensor = mixture_tensor[:max_len]
        else:
            # Pad with zeros if too short
            padding = max_len - mixture_tensor.size(0)
            mixture_tensor = torch.nn.functional.pad(mixture_tensor, (0, padding))
        
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        data = np.load(entry['label_path'], allow_pickle=True)
        audio_labels = torch.from_numpy(data)
        
        label_max_frames = 2994
        max_speakers = 3
        
        current_frames = audio_labels.shape[0]
        if current_frames < label_max_frames:
            pad_frames = label_max_frames - current_frames
            audio_labels = F.pad(audio_labels, (0, 0, 0, pad_frames))  # pad time dim
        elif current_frames > label_max_frames:
            audio_labels = audio_labels[:label_max_frames, :]
        
        current_speakers = audio_labels.shape[1]
        if current_speakers < max_speakers:
            pad_cols = max_speakers - current_speakers
            audio_labels = F.pad(audio_labels, (0, pad_cols))
        
        return mixture_tensor, label_tensor, audio_labels

In [566]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

10000
2000


In [567]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

In [568]:
def torchLMF(audio, n_fft, hop_length):
    torch_lmf = T.MelSpectrogram(
        sample_rate=16000,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=80,
        center=False
        )
    mel_spec = torch_lmf(audio)
    log = torch.log(mel_spec + 1e-8)
    return log

In [569]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.linear_Q = nn.Linear(d_model, d_model)
        self.linear_K = nn.Linear(d_model, d_model)
        self.linear_V = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.linear_Q(x)
        K = self.linear_K(x)
        V = self.linear_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)
        output = self.out_proj(context)
        return output

class TransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = SelfAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self,x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class MiniEEND(nn.Module):
    def __init__(self, n_mels, d_model, num_speakers):
        super().__init__()
        self.linearprojection = nn.Linear(n_mels, d_model)
        self.block1 = TransformerBlock(d_model=d_model)
        self.block2 = TransformerBlock(d_model=d_model)
        self.outputlayer = nn.Linear(d_model, num_speakers)
        

    def forward(self,x):
        x = x.transpose(-1,-2)
        x = self.linearprojection(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.outputlayer(x)
        x = torch.sigmoid(x)
        return x

In [570]:
def align_temporal_dimensions(features, targets):
    """
    Synchronizes the time dimension (T) between features and targets 
    by truncating to the minimum length available in both.
    """
    # Identify time dimensions based on expected shapes:
    # Features: (80, T) or (Batch, 80, T)
    # Targets: (T, Speakers) or (Batch, T, Speakers)
    T_lmf = features.shape[-1]       
    T_label = targets.shape[-2]  
    T_min = min(T_lmf, T_label)          
    
    # Handle single sample (2D features)
    if features.dim() == 2:
        features = features[..., :T_min]         
        targets = targets[:T_min, :]    
    
    # Handle batch (3D features)
    elif features.dim() == 3:
        features = features[..., :T_min]         
        targets = targets[:, :T_min, :]    
    
    return features, targets

### todo 1

part 1

In [571]:
n_fft = 400
hop_length = 160
n_mels = 80
d_model=64
num_speakers=3

audio, num, targets = train_dataset[0]

print(f"audio shape: {audio.shape}")
print(f"number of speakers: {num}")

features = torchLMF(audio, n_fft=n_fft, hop_length=hop_length)
features, targets = align_temporal_dimensions(features, targets)

print(f"targets shape: {targets.shape}")
print(f"input shape: {features.shape}")

model_MiniEEND_single = MiniEEND(n_mels=n_mels, d_model=d_model, num_speakers=num_speakers)
predictions = model_MiniEEND_single(features)

print(f"predictions shape: {predictions.shape}")

audio shape: torch.Size([480000])
number of speakers: 2
targets shape: torch.Size([2994, 3])
input shape: torch.Size([80, 2994])
predictions shape: torch.Size([2994, 3])


In [572]:
def compute_cost_matrix(predictions, targets):
    epsilon = 1e-8
    num_speakers = predictions.shape[-1]
    cost_matrix = torch.zeros(num_speakers, num_speakers)
    
    for i in range(num_speakers):
        for j in range(num_speakers):
            p = predictions[:, i]
            t = targets[:, j]
            cost_matrix[i][j] = -torch.mean(t*torch.log(p+epsilon) + (1-t)*torch.log(1-p+epsilon))
    
    return cost_matrix

In [573]:
print(compute_cost_matrix(predictions, targets))

tensor([[1.5136, 1.8236, 1.5762],
        [0.7701, 0.8560, 0.8012],
        [1.4585, 1.7365, 1.5047]], grad_fn=<CopySlices>)


part 2

In [574]:
def pit_loss_single(predictions, targets):
    cost_matrix = compute_cost_matrix(predictions, targets)
    row_idx, col_idx = scipy.optimize.linear_sum_assignment(cost_matrix.detach().numpy())
    loss = cost_matrix[row_idx, col_idx].mean()
    return loss

In [575]:
pit_loss_single(predictions, targets)

tensor(1.2914, grad_fn=<MeanBackward0>)

part 3

In [576]:
n_fft = 400
hop_length = 160
n_mels = 80
d_model=64
num_speakers=3

batch, num, targets = next(iter(train_loader))

print(f"batch shape: {batch.shape}")
print(f"number of speakers shape: {num.shape}")

features = torchLMF(batch, n_fft=n_fft, hop_length=hop_length)
features, targets = align_temporal_dimensions(features, targets)

print(f"targets shape: {targets.shape}")
print(f"input shape: {features.shape}")

model_MiniEEND_batch = MiniEEND(n_mels=n_mels, d_model=d_model, num_speakers=num_speakers)
predictions = model_MiniEEND_batch(features)

print(f"predictions shape: {predictions.shape}")

batch shape: torch.Size([32, 480000])
number of speakers shape: torch.Size([32])
targets shape: torch.Size([32, 2994, 3])
input shape: torch.Size([32, 80, 2994])
predictions shape: torch.Size([32, 2994, 3])


In [577]:
def pit_loss_batch(predictions, targets):
    if predictions.dim() == 2:
        return pit_loss_single(predictions, targets)
    
    num_batch = predictions.shape[0]
    losses = []
    
    for i in range(num_batch):
        losses.append(pit_loss_single(predictions[i], targets[i]))
    
    return torch.stack(losses).mean()

In [578]:
loss = pit_loss_batch(predictions, targets)
print(loss)

tensor(0.8128, grad_fn=<MeanBackward0>)


part 4

In [579]:
loss.backward()
any(p.grad is not None for p in model_MiniEEND_batch.parameters())

True